# 1. Príprava dát

**Úloha:** rozpoznať z krátkeho videoklipu, či človek **je** (`eat`) alebo **pije** (`drink`).

**HMDB51** (51 činností, klipy z filmov a YouTube), verzia s extrahovanými snímkami:
https://www.kaggle.com/datasets/easonlll/hmdb51 . Z archívu stačí rozbaliť dve triedy:

```bash
unzip -q hmdb51.zip 'HMDB51/eat/*' 'HMDB51/drink/*' -d .
```

Voliteľne sa pridajú klipy z **Charades** (notebook 0, súbor `charades_clips.csv`).

V tomto notebooku:
1. vytvoríme spoločný zoznam klipov (`index.csv`),
2. ukážeme, ako delíme dáta na trénovacie a testovacie,
3. pozrieme sa na dĺžku klipov a na ukážku snímok.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from har_utils import CLASSES, make_splits, read_frames

ROOT = Path("HMDB51")

## Zoznam klipov

Každý klip má:
- `label` – číslo triedy (0 = eat, 1 = drink),
- `source` – zdroj klipu pre delenie dát: pri HMDB51 film (časť názvu klipu pred názvom
  triedy, napr. `April_09_brush_hair_u_nm_np1_ba_goo_0` → `April_09`), pri Charades človek,
- `dataset` – `hmdb` alebo `charades`.

In [ ]:
rows = []
for cls in CLASSES:
    for clip in sorted((ROOT / cls).iterdir()):
        if clip.is_dir():
            # názov klipu: <zdroj>_<trieda>_<6 polí metadát>
            source = clip.name.rsplit("_", 6)[0].removesuffix(f"_{cls}")
            rows.append((str(clip), cls, source, "hmdb"))
df = pd.DataFrame(rows, columns=["path", "cls", "source", "dataset"])

if Path("charades_clips.csv").exists():                # klipy z notebooku 0
    ch = pd.read_csv("charades_clips.csv").assign(dataset="charades")
    df = pd.concat([df, ch], ignore_index=True)

df["label"] = df.cls.map(CLASSES.index)
df = df[["path", "cls", "label", "source", "dataset"]]
df.to_csv("index.csv", index=False)

print(pd.crosstab(df.dataset, df.cls, margins=True))
print("počet zdrojov (filmov / ľudí):", df.groupby("dataset").source.nunique().to_dict())

## Delenie na trénovacie a testovacie dáta

Z jedného filmu je v HMDB51 často vystrihnutých viac podobných klipov (a v Charades
nahral jeden človek viac videí). Keby sme delili náhodne po klipoch, podobné zábery by
sa dostali do tréningu aj do testu a presnosť by bola nadhodnotená (**únik dát**).
Preto delíme **podľa zdroja**: všetky klipy z jedného filmu / od jedného človeka sú
buď v train, alebo v test.

Jeden testovací výsledok na malom datasete je veľmi zašumený, preto každý model
hodnotíme na **10 rôznych deleniach** 70/30 a uvádzame priemer ± smerodajnú odchýlku.
Výsledky uvádzame zvlášť pre testovacie klipy z HMDB51 a z Charades.
Funkcia `make_splits` je v `har_utils.py`. Ukážka prvého delenia:

In [ ]:
seed, tr, te = next(make_splits(df))
part = pd.Series("train", index=df.index, name="split")
part.iloc[te] = "test"
print(pd.crosstab([df.dataset, part], df.cls))

## Dĺžka klipov

Z každého klipu neskôr vyberáme 16 rovnomerne rozložených snímok.

In [ ]:
df["n_frames"] = df.path.map(lambda p: len(list(Path(p).glob("*.jpg"))))
print(df.groupby("dataset").n_frames.describe().round(1))

## Ukážka klipov

Takto vyzerá vstup modelu: 16 snímok rovnomerne vybraných z klipu.

In [ ]:
for (dataset, cls), g in df.groupby(["dataset", "cls"]):
    row = g.iloc[0]
    frames = read_frames(row.path)            # (16, 224, 224, 3)
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    for ax, f in zip(axes.flat, frames):
        ax.imshow(f)
        ax.axis("off")
    fig.suptitle(f"{dataset} / {cls}: {Path(row.path).name}")
    plt.show()